# Populations for three test simulations

This notebook reproduces Figure 5 in [Graber et al. (2024)](https://arxiv.org/abs/2312.14848).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from scipy import stats

from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.basics.constants as const
import mlpoppyns.generator.maps.axes_scaling as axs
import utilities.plot_settings

from matplotlib import rc

rc("text", usetex=True)
rc("font", family="serif")
mpl.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [ ]:
SMALL_SIZE = 30
MEDIUM_SIZE = 40
BIGGER_SIZE = 50

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=BIGGER_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=BIGGER_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

## Load one simulation

Import three simulated surveys.

In [ ]:
# Comment/uncomment lines below and rerun the entire notebook to produce PPdot plots for a different test sample.

test_sample = 314335
test_no = 1

# test_sample = 298404
# test_no = 2

# test_sample = 310306
# test_no = 3

In [ ]:
# Select the `.pkl.gz` files containing the survey results to import.

df_PMPS_sim = pd.read_pickle(
    f"../../data/paper_results/graber_etal_2024/test_simulations/survey_PMPS_results{test_sample}.pkl.gz",
    compression="gzip",
)

df_SMPS_sim = pd.read_pickle(
    f"../../data/paper_results/graber_etal_2024/test_simulations/survey_SMPS_results{test_sample}.pkl.gz",
    compression="gzip",
)

df_HTRU_sim = pd.read_pickle(
    f"../../data/paper_results/graber_etal_2024/test_simulations/survey_HTRU_results{test_sample}.pkl.gz",
    compression="gzip",
)

df_PMPS_sim.head()

In [ ]:
# Extracting the parameters.
l_pk_sim = df_PMPS_sim["l"]["[deg]"].to_numpy()
b_pk_sim = df_PMPS_sim["b"]["[deg]"].to_numpy()
P_pk_sim = df_PMPS_sim["P"]["[s]"].to_numpy()
Pdot_pk_sim = df_PMPS_sim["P_dot"]["[s s^-1]"].to_numpy()

In [ ]:
l_sw_sim = df_SMPS_sim["l"]["[deg]"].to_numpy()
b_sw_sim = df_SMPS_sim["b"]["[deg]"].to_numpy()
P_sw_sim = df_SMPS_sim["P"]["[s]"].to_numpy()
Pdot_sw_sim = df_SMPS_sim["P_dot"]["[s s^-1]"].to_numpy()

In [ ]:
l_htru_sim = df_HTRU_sim["l"]["[deg]"].to_numpy()
b_htru_sim = df_HTRU_sim["b"]["[deg]"].to_numpy()
P_htru_sim = df_HTRU_sim["P"]["[s]"].to_numpy()
Pdot_htru_sim = df_HTRU_sim["P_dot"]["[s s^-1]"].to_numpy()

## Plotting the observations

In [ ]:
colors = ["#FFAC1C", "dodgerblue", "#440154"]

Edot lines.

In [ ]:
I_NS = 1.36e45
R_NS = 1.1e6
c = 2.998e10

Pdot_Edot_lines = np.zeros((6, 51))

Edot_log = np.linspace(28, 38, 6)
print(Edot_log)

In [ ]:
P_log = np.linspace(-3, 2, 51)

In [ ]:
for i in range(len(Edot_log)):
    Pdot_Edot_lines[i] = (
        10 ** Edot_log[i] * (10**P_log) ** 3 / (4 * np.pi**2 * I_NS)
    )

B lines.

In [ ]:
Pdot_B_lines = np.zeros((5, 51))

B_log = np.linspace(10, 14, 5)
print(B_log)

In [ ]:
for i in range(len(B_log)):
    Pdot_B_lines[i] = (
        np.pi**2
        * (10 ** B_log[i]) ** 2
        * (R_NS**6)
        / (I_NS * 10**P_log * c**3)
    )

PPdot diagram.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 10))

for i in range(len(Edot_log)):
    ax.plot(
        10**P_log,
        Pdot_Edot_lines[i],
        linestyle="-",
        color="gray",
        alpha=0.5,
        rasterized=True,
    )
for i in range(len(B_log)):
    ax.plot(
        10**P_log,
        Pdot_B_lines[i],
        linestyle="-",
        color="gray",
        alpha=0.5,
        rasterized=True,
    )
ax.plot(
    P_pk_sim,
    Pdot_pk_sim,
    linestyle="None",
    marker="o",
    color=colors[0],
    markersize=9,
    alpha=1.0,
    rasterized=True,
    label="Simulated PMPS",
)
ax.plot(
    P_sw_sim,
    Pdot_sw_sim,
    linestyle="None",
    marker="o",
    color=colors[1],
    markersize=9,
    alpha=1.0,
    rasterized=True,
    label="Simulated SMPS",
)
ax.plot(
    P_htru_sim,
    Pdot_htru_sim,
    linestyle="None",
    marker="o",
    color=colors[2],
    markersize=9,
    alpha=0.3,
    rasterized=True,
    label="Simulated HTRU",
)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1.0e-3, 100.0)
ax.set_ylim(1.0e-21, 1.0e-9)
ax.set_title(f"Test sample {test_no}", fontsize=BIGGER_SIZE)

ax.text(
    0.41,
    0.015,
    r"$10^{28} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.275,
    0.015,
    r"$10^{30} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.143,
    0.015,
    r"$10^{32} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.015,
    r"$10^{34} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.181,
    r"$10^{36} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.01,
    0.347,
    r"$10^{38} \, {\rm erg} \, {\rm s}^{-1}$",
    rotation=50,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)

ax.text(
    0.89,
    0.005,
    r"$10^{10} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.172,
    r"$10^{11} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.340,
    r"$10^{12} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.506,
    r"$10^{13} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)
ax.text(
    0.89,
    0.672,
    r"$10^{14} \, {\rm G}$",
    rotation=-23,
    fontsize=20,
    transform=ax.transAxes,
    color="gray",
)

plt.xlabel(r"Period $P$ [s]")
plt.ylabel(r"Period derivative $\dot{P}$ [s$\,{\rm s}^{-1}$]")
ax.legend(frameon=True, loc=2)

plt.tight_layout()
plt.savefig(
    f"../../paper_plots/graber_etal_2024/plots/simulated_ppdot_{test_no}.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## Density maps

Set plotting parameters.

In [ ]:
resolution = 32

In [ ]:
n_x_bins = resolution
n_y_bins = resolution

x_range = (0.001, 100.0)
y_range = (1.0e-21, 1.0e-9)

x_log_scale = True
y_log_scale = True

x_edges, y_edges = axs.log_scale_vs_linear_scale(
    x_range,
    y_range,
    x_log_scale,
    y_log_scale,
    n_x_bins,
    n_y_bins,
)

colormap = "viridis"
print(y_edges)

Load the .npy file for test sample 1 outputted from the generator.

In [ ]:
density_ppdot_PMPS = np.load(
    "../../data/paper_results/graber_etal_2024/test_simulations/survey_PMPS_ppdot_map_314335.npy"
)
density_ppdot_SMPS = np.load(
    "../../data/paper_results/graber_etal_2024/test_simulations/survey_SMPS_ppdot_map_314335.npy"
)
density_ppdot_HTRU = np.load(
    "../../data/paper_results/graber_etal_2024/test_simulations/survey_HTRU_ppdot_map_314335.npy"
)

# Check resolution is correct
print(np.shape(density_ppdot_PMPS))
print(np.shape(density_ppdot_SMPS))
print(np.shape(density_ppdot_HTRU))

Make density maps.

In [ ]:
fig = plt.figure(frameon=False)

fig.set_size_inches(n_x_bins / 5, n_y_bins / 5)
ax = plt.Axes(fig, [0.0, 0.0, 1.0, 1.0])
ax.set_axis_off()
fig.add_axes(ax)

# Generating a pseudocolor plot of the smeared out density distribution;
# we transpose the array as pcolormesh is indexed starting from the lower
# left , i.e., the column (row) index corresponds to the x (y) coordinate.
ax.pcolormesh(x_edges, y_edges, density_ppdot_PMPS.T, cmap=colormap)
ax.set_xlim(x_range[0], x_range[1])
ax.set_ylim(y_range[0], y_range[1])
ax.set_title(f"Test sample 1: PMPS", fontsize=MEDIUM_SIZE)

if x_log_scale:
    ax.set_xscale("log")

if y_log_scale:
    ax.set_yscale("log")

plt.savefig(
    f"../../paper_plots/graber_etal_2024/plots/simulated_ppdot_PMPS.pdf",
    dpi=400,
    bbox_inches="tight",
)
plt.show()

In [ ]:
fig = plt.figure(frameon=False)

fig.set_size_inches(n_x_bins / 5, n_y_bins / 5)
ax = plt.Axes(fig, [0.0, 0.0, 1.0, 1.0])
ax.set_title(f"Test sample 1: SMPS", fontsize=MEDIUM_SIZE)
ax.set_axis_off()
fig.add_axes(ax)

# Generating a pseudocolor plot of the smeared out density distribution;
# we transpose the array as pcolormesh is indexed starting from the lower
# left , i.e., the column (row) index corresponds to the x (y) coordinate.
ax.pcolormesh(x_edges, y_edges, density_ppdot_SMPS.T, cmap=colormap)
ax.set_xlim(x_range[0], x_range[1])
ax.set_ylim(y_range[0], y_range[1])

if x_log_scale:
    ax.set_xscale("log")

if y_log_scale:
    ax.set_yscale("log")

plt.savefig(
    f"../../paper_plots/graber_etal_2024/plots/simulated_ppdot_SMPS.pdf",
    dpi=400,
    bbox_inches="tight",
)
plt.show()

In [ ]:
fig = plt.figure(frameon=False)

fig.set_size_inches(n_x_bins / 5, n_y_bins / 5)
ax = plt.Axes(fig, [0.0, 0.0, 1.0, 1.0])
ax.set_axis_off()
fig.add_axes(ax)

# Generating a pseudocolor plot of the smeared out density distribution;
# we transpose the array as pcolormesh is indexed starting from the lower
# left , i.e., the column (row) index corresponds to the x (y) coordinate.
ax.pcolormesh(x_edges, y_edges, density_ppdot_HTRU.T, cmap=colormap)
ax.set_xlim(x_range[0], x_range[1])
ax.set_ylim(y_range[0], y_range[1])
ax.set_title(f"Test sample 1: HTRU", fontsize=MEDIUM_SIZE)

if x_log_scale:
    ax.set_xscale("log")

if y_log_scale:
    ax.set_yscale("log")

plt.savefig(
    f"../../paper_plots/graber_etal_2024/plots/simulated_ppdot_HTRU.pdf",
    dpi=400,
    bbox_inches="tight",
)
plt.show()